# Python practice: Measure conversion from messy checkout events

You have a small fictional event log with repeat visits and retried payments. Use Python sets to calculate the share of unique visitors who bought through each channel.

Run this notebook from top to bottom in Jupyter with a Python 3 kernel. The data and solution are included. The exercise uses Python’s standard library.

In [ ]:
from csv import DictReader
from io import StringIO

events = list(DictReader(StringIO("""channel,visitor,event
search,a,visit
search,a,visit
search,b,visit
search,c,visit
search,a,paid
search,a,paid
search,b,declined
email,d,visit
email,e,visit
email,d,paid
email,f,paid
""")))
channels = ("search", "email", "partner")

## Your task

1. For each channel, count unique visitors with a `visit` event.
2. Count unique buyers who also appear among that channel’s visitors. A duplicated `paid` event counts once. An unmatched payment does not enter that channel’s conversion rate.
3. Divide buyers by visitors. Use `None` for a channel with zero visitors.

Set aside the solution until you can predict the results. The expected visitor and buyer counts are **3 and 1** for Search, **2 and 1** for Email, and **0 and 0** for Partner.

## Worked solution

Keep a set of visitors and a set of people with a paid event per channel. Their intersection gives the buyers eligible for the conversion numerator. This example reports unique-buyer conversion within the supplied event log.

In [ ]:
def channel_conversion(rows, channel_names):
    visits = {name: set() for name in channel_names}
    purchases = {name: set() for name in channel_names}
    for row in rows:
        channel = row["channel"]
        if channel not in visits:
            continue
        if row["event"] == "visit":
            visits[channel].add(row["visitor"])
        elif row["event"] == "paid":
            purchases[channel].add(row["visitor"])

    result = {}
    for name in visits:
        visitors = len(visits[name])
        buyers = len(visits[name] & purchases[name])
        rate = buyers / visitors if visitors else None
        result[name] = {"visitors": visitors, "buyers": buyers, "rate": rate}
    return result


results = channel_conversion(events, channels)
for name, row in results.items():
    rate = "unmeasured" if row["rate"] is None else f'{row["rate"]:.1%}'
    print(f'{name:8} {row["buyers"]}/{row["visitors"]}  {rate}')

In [ ]:
assert results == {
    "search": {"visitors": 3, "buyers": 1, "rate": 1 / 3},
    "email": {"visitors": 2, "buyers": 1, "rate": 1 / 2},
    "partner": {"visitors": 0, "buyers": 0, "rate": None},
}
assert channel_conversion(events + events, channels) == results
assert channel_conversion(events[::-1], channels) == results
assert channel_conversion([], ("search",))["search"]["rate"] is None

## Keep practicing

This is a standalone preview exercise. The paid Python for Data Science workbook includes ten practice notebooks, worked solutions, datasets, and a setup guide.

[See the Python workbook and its full module list on Gumroad](https://diego612.gumroad.com/l/bmkmqv?utm_source=github&utm_medium=organic&utm_campaign=python-workbook-preview).